<a href="https://colab.research.google.com/github/ga426553-sudo/Classification-of-Breast-Cancer-Subtypes-machine-learning---CuMiDa-22820/blob/main/No_6_RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Sexto código - Cáncer de Mama 🌸

**Implementación de Random Forest para Clasificación de Cáncer de Mama**

###Importar librerías necesarias 🌺

In [ ]:
# ============================================
# CÓDIGO 6: RANDOM FOREST Y EVALUACIÓN
# ============================================

from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("="*60)
print("🌲 CÓDIGO 7: EVALUACIÓN DE RANDOM FOREST")
print("="*60)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🌲 CÓDIGO 7: EVALUACIÓN DE RANDOM FOREST


###Cargar datos 🌺

In [ ]:
# 1. CARGAR DATOS
# ================
print(f"\n📂 Cargando datos del Código 2...")

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)

y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
y_test = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_test_final.npy')

print(f"✅ Datos cargados:")
print(f"   Subconjuntos: {list(subsets.keys())}")
print(f"   y_train: {y_train.shape}")
print(f"   y_test: {y_test.shape}")


📂 Cargando datos del Código 2...
✅ Datos cargados:
   Subconjuntos: ['EO', 'BBA', 'CSA', 'RDA', 'GA']
   y_train: (206,)
   y_test: (28,)


###Configuración de random forest 🌺

In [ ]:
# 2. CONFIGURAR RANDOM FOREST (basado en el artículo, similar a XGBoost)
# ========================================================================
rf_params = {
    'n_estimators': 100,
    'max_depth': 4,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'criterion': 'gini',
    'random_state': 42
}

rf_model = RandomForestClassifier(**rf_params)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print(f"\n⚙️ Configuración de Random Forest:")
for key, value in rf_params.items():
    print(f"   {key}: {value}")


⚙️ Configuración de Random Forest:
   n_estimators: 100
   max_depth: 4
   min_samples_split: 2
   min_samples_leaf: 1
   criterion: gini
   random_state: 42


### Evaluación de Random Forest 🌺

In [ ]:
# 3. EVALUAR CADA SUBCONJUNTO
# ============================
results = {}

for name in subsets.keys():
    print(f"\n{'─'*40}")
    print(f"🔍 Subconjunto: {name} - Genes: {subsets[name]}")

    X_train_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{name}.npy')
    X_test_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/test_{name}.npy')

    # Validación cruzada (10-fold)
    cv_acc = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='accuracy')
    cv_f1 = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='f1')
    cv_auc = cross_val_score(rf_model, X_train_subset, y_train, cv=cv, scoring='roc_auc')

    # Entrenar modelo completo
    rf_model.fit(X_train_subset, y_train)

    # Predicciones
    y_pred = rf_model.predict(X_test_subset)
    y_proba = rf_model.predict_proba(X_test_subset)[:, 1]

    # Métricas en test
    test_acc = accuracy_score(y_test, y_pred)
    test_f1 = f1_score(y_test, y_pred)
    test_auc = roc_auc_score(y_test, y_proba)

    # Importancia de características
    feature_importance = pd.DataFrame({
        'gene': subsets[name],
        'importance': rf_model.feature_importances_
    }).sort_values('importance', ascending=False)

    results[name] = {
        'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
        'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
        'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
        'test_accuracy': test_acc,
        'test_f1': test_f1,
        'test_auc': test_auc,
        'genes': subsets[name],
        'feature_importance': feature_importance,
        'y_pred': y_pred,
        'y_proba': y_proba
    }

    print(f"\n   Resultados:")
    print(f"   📊 Validación Cruzada (10-fold):")
    print(f"      Accuracy: {results[name]['cv_accuracy']}")
    print(f"      F1-Score: {results[name]['cv_f1']}")
    print(f"      AUC: {results[name]['cv_auc']}")
    print(f"\n   📈 Test:")
    print(f"      Accuracy: {test_acc:.4f}")
    print(f"      F1-Score: {test_f1:.4f}")
    print(f"      AUC: {test_auc:.4f}")
    print(f"\n   🔬 Importancia de genes:")
    print(feature_importance.to_string(index=False))


────────────────────────────────────────
🔍 Subconjunto: EO - Genes: ['NM_138957', 'NM_152426', 'NM_001008493']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9471 ± 0.0688
      F1-Score: 0.9450 ± 0.0699
      AUC: 0.9918 ± 0.0165

   📈 Test:
      Accuracy: 0.9286
      F1-Score: 0.9615
      AUC: 0.8846

   🔬 Importancia de genes:
        gene  importance
   NM_138957    0.459916
NM_001008493    0.312449
   NM_152426    0.227635

────────────────────────────────────────
🔍 Subconjunto: BBA - Genes: ['NM_152426', 'NM_138957']

   Resultados:
   📊 Validación Cruzada (10-fold):
      Accuracy: 0.9181 ± 0.0568
      F1-Score: 0.9158 ± 0.0572
      AUC: 0.9675 ± 0.0255

   📈 Test:
      Accuracy: 0.9286
      F1-Score: 0.9615
      AUC: 0.9038

   🔬 Importancia de genes:
     gene  importance
NM_138957    0.666971
NM_152426    0.333029

────────────────────────────────────────
🔍 Subconjunto: CSA - Genes: ['BC016934', 'NM_006579']

   Resultados:
   📊 Validación Cruza

###Resultados 🌺

In [ ]:
# 4. MOSTRAR RESULTADOS COMPARATIVOS
# ===================================
print(f"\n{'='*60}")
print("📊 RESULTADOS FINALES - RANDOM FOREST")
print('='*60)

comparison_df = pd.DataFrame({
    'Subset': results.keys(),
    'Genes': [', '.join(r['genes']) for r in results.values()],
    'Test_Accuracy': [f"{r['test_accuracy']:.4f}" for r in results.values()],
    'Test_F1': [f"{r['test_f1']:.4f}" for r in results.values()],
    'Test_AUC': [f"{r['test_auc']:.4f}" for r in results.values()],
    'CV_Accuracy': [r['cv_accuracy'] for r in results.values()]
})

print("\n📋 Tabla comparativa:")
print(comparison_df.to_string(index=False))


📊 RESULTADOS FINALES - RANDOM FOREST

📋 Tabla comparativa:
Subset                              Genes Test_Accuracy Test_F1 Test_AUC     CV_Accuracy
    EO NM_138957, NM_152426, NM_001008493        0.9286  0.9615   0.8846 0.9471 ± 0.0688
   BBA               NM_152426, NM_138957        0.9286  0.9615   0.9038 0.9181 ± 0.0568
   CSA                BC016934, NM_006579        0.8571  0.9200   0.6923 0.9274 ± 0.0449
   RDA                           BC016934        0.5714  0.7143   0.5769 0.8295 ± 0.0564
    GA                          NM_152426        0.8214  0.8936   0.9327 0.7290 ± 0.0856


In [ ]:
# ============================================
# BASELINE: Random Forest con TODOS los genes
# ============================================

print("\n" + "="*60)
print("📊 BASELINE - Random Forest con TODOS LOS GENES")
print("="*60)

X_train_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy')
X_test_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_test_clean.npy')

print(f"Datos baseline: {X_train_all.shape[1]} genes")
print("⚠️ Esto puede tomar varios minutos (10-fold CV con ~2000 genes)...")

rf_baseline = RandomForestClassifier(
    n_estimators=100,
    max_depth=4,
    min_samples_split=2,
    min_samples_leaf=1,
    criterion='gini',
    random_state=42,
    n_jobs=-1  # Usar todos los núcleos para acelerar
)

cv_baseline = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

print("\n🔄 Ejecutando validación cruzada (10-fold)...")
cv_acc = cross_val_score(rf_baseline, X_train_all, y_train, cv=cv_baseline, scoring='accuracy', n_jobs=-1)
cv_f1 = cross_val_score(rf_baseline, X_train_all, y_train, cv=cv_baseline, scoring='f1', n_jobs=-1)
cv_auc = cross_val_score(rf_baseline, X_train_all, y_train, cv=cv_baseline, scoring='roc_auc', n_jobs=-1)

print(f"\n📊 Resultados Baseline Random Forest:")
print(f"   CV Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"   CV F1-Score: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")
print(f"   CV AUC: {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")

rf_baseline.fit(X_train_all, y_train)
y_pred = rf_baseline.predict(X_test_all)
y_proba = rf_baseline.predict_proba(X_test_all)[:, 1]

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_auc = roc_auc_score(y_test, y_proba)

print(f"\n📊 Test Baseline Random Forest:")
print(f"   Test Accuracy: {test_acc:.4f}")
print(f"   Test F1-Score: {test_f1:.4f}")
print(f"   Test AUC: {test_auc:.4f}")

baseline_results = {
    'cv_accuracy': f"{cv_acc.mean():.4f} ± {cv_acc.std():.4f}",
    'cv_f1': f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
    'cv_auc': f"{cv_auc.mean():.4f} ± {cv_auc.std():.4f}",
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'test_auc': test_auc
}

with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/baseline_rf.pkl', 'wb') as f:
    pickle.dump(baseline_results, f)

print("\n✅ Baseline Random Forest guardado")


📊 BASELINE - Random Forest con TODOS LOS GENES
Datos baseline: 31215 genes
⚠️ Esto puede tomar varios minutos (10-fold CV con ~2000 genes)...

🔄 Ejecutando validación cruzada (10-fold)...

📊 Resultados Baseline Random Forest:
   CV Accuracy: 1.0000 ± 0.0000
   CV F1-Score: 1.0000 ± 0.0000
   CV AUC: 1.0000 ± 0.0000

📊 Test Baseline Random Forest:
   Test Accuracy: 1.0000
   Test F1-Score: 1.0000
   Test AUC: 1.0000

✅ Baseline Random Forest guardado


### Evaluación con Validación Cruzada Repetida (Decision Tree y Random Forest) 🌸

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
import numpy as np
import pickle
from google.colab import drive # Import drive module

# Ensure drive is mounted, as this can sometimes be flaky
drive.mount('/content/drive')

# Cargar datos (ajusta la ruta si es necesario)
with open('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/subsets.pkl', 'rb') as f:
    subsets = pickle.load(f)
y_train = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/y_train_final.npy')
X_train_all = np.load('/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/X_train_balanced.npy')

# Configuración de modelos (parámetros del artículo)
dt = DecisionTreeClassifier(criterion='gini', max_depth=4, min_samples_split=2, min_samples_leaf=1, random_state=42)
rf = RandomForestClassifier(n_estimators=100, max_depth=4, min_samples_split=2, min_samples_leaf=1, criterion='gini', random_state=42, n_jobs=-1)

# CV repetida
rkf = RepeatedStratifiedKFold(n_splits=10, n_repeats=5, random_state=42)

print("="*60)
print("RESULTADOS CON CV REPETIDA (10 folds, 5 repeticiones)")
print("="*60)

for model, name_model in zip([dt, rf], ['Decision Tree', 'Random Forest']):
    print(f"\n🔹 {name_model}")
    print("-"*40)
    for subset_name in subsets.keys():
        X_subset = np.load(f'/content/drive/MyDrive/Octavo semestre/Optativa2/BreastCancer/train_{subset_name}.npy')
        acc = cross_val_score(model, X_subset, y_train, cv=rkf, scoring='accuracy')
        f1 = cross_val_score(model, X_subset, y_train, cv=rkf, scoring='f1')
        auc = cross_val_score(model, X_subset, y_train, cv=rkf, scoring='roc_auc')
        print(f"{subset_name}: Acc = {acc.mean():.4f} ± {acc.std():.4f}, "
              f"F1 = {f1.mean():.4f} ± {f1.std():.4f}, "
              f"AUC = {auc.mean():.4f} ± {auc.std():.4f}")
    # Baseline (todos los genes)
    acc_all = cross_val_score(model, X_train_all, y_train, cv=rkf, scoring='accuracy')
    f1_all = cross_val_score(model, X_train_all, y_train, cv=rkf, scoring='f1')
    auc_all = cross_val_score(model, X_train_all, y_train, cv=rkf, scoring='roc_auc')
    print(f"Baseline: Acc = {acc_all.mean():.4f} ± {acc_all.std():.4f}, "
          f"F1 = {f1_all.mean():.4f} ± {f1_all.std():.4f}, "
          f"AUC = {auc_all.mean():.4f} ± {auc_all.std():.4f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
RESULTADOS CON CV REPETIDA (10 folds, 5 repeticiones)

🔹 Decision Tree
----------------------------------------
EO: Acc = 0.9263 ± 0.0459, F1 = 0.9247 ± 0.0472, AUC = 0.9481 ± 0.0460
BBA: Acc = 0.9080 ± 0.0576, F1 = 0.9066 ± 0.0583, AUC = 0.9228 ± 0.0604
CSA: Acc = 0.9303 ± 0.0484, F1 = 0.9274 ± 0.0503, AUC = 0.9516 ± 0.0475
RDA: Acc = 0.8067 ± 0.0666, F1 = 0.7762 ± 0.0841, AUC = 0.8192 ± 0.0817
GA: Acc = 0.7250 ± 0.1043, F1 = 0.6865 ± 0.1528, AUC = 0.8085 ± 0.0912
Baseline: Acc = 0.9923 ± 0.0242, F1 = 0.9919 ± 0.0261, AUC = 0.9925 ± 0.0232

🔹 Random Forest
----------------------------------------
EO: Acc = 0.9486 ± 0.0464, F1 = 0.9465 ± 0.0483, AUC = 0.9940 ± 0.0118
BBA: Acc = 0.9148 ± 0.0537, F1 = 0.9126 ± 0.0559, AUC = 0.9742 ± 0.0275
CSA: Acc = 0.9206 ± 0.0491, F1 = 0.9190 ± 0.0498, AUC = 0.9616 ± 0.0456
RDA: Acc = 0.8321 ± 0.0607, F1 = 0.8044 ± 0.0782, A